# 06.01 · Prompting fundamentals

**Chapter:** Prompt & Context Engineering · **Milestone:** M2

## Goals
- [ ] Apply the core techniques: clear task, definitions, examples (few-shot), delimiters / XML tags, output format
- [ ] Treat prompt changes as experiments: one change at a time, measured on a labelled set
- [ ] Read errors, not just the score

## Resources
- [Anthropic – Interactive prompt engineering tutorial](https://github.com/anthropics/prompt-eng-interactive-tutorial) — free
- [Anthropic – Prompt engineering overview](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview) · [OpenAI – Prompting guide](https://platform.openai.com/docs/guides/text) — free
- [Prompting Guide](https://www.promptingguide.ai/) — free

The task: route 30 support tickets (`tickets.json`) into 5 categories. Each version below changes **one thing**.
Numbers come from a small local model (qwen3:1.7b) — bigger models score higher, but the *relative* lessons hold.

In [ ]:
import json
from collections import Counter
from pathlib import Path

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
tickets = json.loads(Path("tickets.json").read_text())
CATEGORIES = ["billing", "bug", "feature_request", "account", "other"]
print(len(tickets), "tickets |", Counter(t["label"] for t in tickets))

## The experiment harness
Run a prompt over every ticket, parse the answer, score it, keep the failures. You will reuse this pattern forever (chapter 10).

In [ ]:
def parse_label(text):
    text = text.strip().lower().replace(" ", "_").replace("-", "_")
    hits = [cat for cat in CATEGORIES if cat in text]
    return hits[0] if len(hits) == 1 else "UNPARSEABLE"


def run(build_messages, name):
    results = []
    for t in tickets:
        try:  # max_tokens: always cap output — a rambling or looping model costs money and time
            out = c.chat.completions.create(model=MODEL, messages=build_messages(t["text"]), temperature=0, max_tokens=200, **EXTRA).choices[0].message.content
        except Exception as e:  # e.g. local server aborting a repetition loop
            out = f"ERROR: {e}"
        results.append({**t, "raw": out, "pred": parse_label(out)})
    acc = sum(r["pred"] == r["label"] for r in results) / len(results)
    unparseable = sum(r["pred"] == "UNPARSEABLE" for r in results)
    print(f"{name:32} accuracy {acc:.0%} | unparseable {unparseable}")
    return results


scoreboard = {}

## v1 — the lazy prompt

In [ ]:
def v1(ticket):
    return [{"role": "user", "content": f"What category is this support ticket? {ticket}"}]

scoreboard["v1 lazy"] = run(v1, "v1 lazy")
print(scoreboard["v1 lazy"][0]["raw"][:300])

Without the list of categories the model invents its own and answers in prose: almost nothing parses, and small models may
even ramble until `max_tokens` (or loop until the server aborts — check for `ERROR` in `raw`).

## v2 — state the task and the allowed labels

In [ ]:
def v2(ticket):
    return [{"role": "user", "content": f"Classify this support ticket into one of: {', '.join(CATEGORIES)}.\n\nTicket: {ticket}"}]

scoreboard["v2 + labels"] = run(v2, "v2 + labels")

## v3 — define the labels and constrain the output
Ambiguity lives in the label boundaries (is "delete my account" *account* or *other*? is "card declined" *billing* or *bug*?).
Definitions resolve it. A system prompt holds the stable instructions; the user turn holds the data.

In [ ]:
DEFINITIONS = {
    "billing": "charges, invoices, refunds, payment methods, pricing, plan changes",
    "bug": "something in the product is broken or behaves incorrectly",
    "feature_request": "asking for new functionality or integrations that don't exist yet",
    "account": "login, password, email, verification, account deletion, ownership, locked accounts",
    "other": "anything else: general questions, company info, compliance, praise, off-topic",
}
SYSTEM_V3 = "You are a support-ticket router.\nCategories:\n" + "\n".join(f"- {k}: {v}" for k, v in DEFINITIONS.items()) + \
            "\n\nAnswer with exactly one category name and nothing else."

def v3(ticket):
    return [{"role": "system", "content": SYSTEM_V3}, {"role": "user", "content": ticket}]

scoreboard["v3 + definitions"] = run(v3, "v3 + definitions + format")

## v4 — few-shot examples, delimited with XML tags
Examples teach format and boundaries better than any description. Rules:
- Cover every label, especially the confusing boundaries.
- **Never use your eval items as examples** (that's leakage, 02.01). These are new, hand-written ones.
- Delimit data with tags (`<ticket>…</ticket>`) so the model can't confuse data with instructions (also a prompt-injection defence, chapter 11).

In [ ]:
FEW_SHOT = [
    ("Why did my bill go up after I added a seat?", "billing"),
    ("The chart legend overlaps the axis labels on mobile.", "bug"),
    ("Would be nice to have a Zapier integration.", "feature_request"),
    ("I want to close my account permanently.", "account"),
    ("What are your support hours?", "other"),
]
SYSTEM_V4 = SYSTEM_V3 + "\n\nExamples:\n" + "\n".join(f"<ticket>{t}</ticket> -> {l}" for t, l in FEW_SHOT)

def v4(ticket):
    return [{"role": "system", "content": SYSTEM_V4}, {"role": "user", "content": f"<ticket>{ticket}</ticket>"}]

scoreboard["v4 + few-shot"] = run(v4, "v4 + few-shot + XML")

## v5 — structured output instead of parsing
Remove the parsing failure mode entirely (05.03).

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Route(BaseModel):
    category: Literal["billing", "bug", "feature_request", "account", "other"]


def run_structured(name):
    results = []
    for t in tickets:
        r = c.chat.completions.parse(model=MODEL, messages=v4(t["text"]), response_format=Route, temperature=0, **EXTRA)
        results.append({**t, "raw": "", "pred": r.choices[0].message.parsed.category})
    print(f"{name:32} accuracy {sum(r['pred'] == r['label'] for r in results) / len(results):.0%} | unparseable 0")
    return results

scoreboard["v5 structured"] = run_structured("v5 few-shot + structured output")

## Read the failures
The score tells you *how much*; the errors tell you *what to change next*.

In [ ]:
best = max(scoreboard, key=lambda k: sum(r["pred"] == r["label"] for r in scoreboard[k]))
print("best:", best)
for r in scoreboard[best]:
    if r["pred"] != r["label"]:
        print(f"  gold={r['label']:16} pred={r['pred']:16} {r['text']}")

## Takeaways
1. **Specify the task fully**: labels, definitions, output format. Most "the model is dumb" problems are underspecified prompts.
2. **Examples** are the strongest lever, but keep them out of your eval set.
3. **Structure** the output instead of parsing prose.
4. **Measure every change.** With 30 items, one ticket = 3.3 points: small differences are noise (02.03's confidence intervals).

Other techniques you'll use: role/persona in the system prompt, "think first" (06.02), prefilling the assistant's answer
(Anthropic), putting long documents *before* the question, and asking the model to quote evidence before answering.

## Exercises
1. Write v6 that fixes the remaining errors without adding the eval tickets as examples. Did you overfit to these 30?
   Write 10 new tickets as a held-out set and check.
2. Run v3 and v4 with a different provider (Gemini/Groq). How much of the gap closes with a bigger model?
3. Shuffle the order of the few-shot examples 3 times. How much does accuracy move? (Order sensitivity is real, especially in small models.)

In [ ]:
# your code here